# SGX REIT -> `sgx_manual_input` upsert

**For non-coders. Follow the steps. You only edit STEP 1.**

1. Put your REIT Excel workbook(s) + the `.env` file in this folder.
2. Run every cell top to bottom (*Run all*).

Each REIT is one **sheet/tab**. The notebook reads every sheet, pulls the property
portfolio from the `sgx_reit_property` table in the database, checks the numbers, shows a
preview, then upserts.

### `financial_year` rule
Exact statement date is kept in `date`. `financial_year` = declared FY:
- ends **Jan-Jun** of year X -> `X - 1`
- ends **Jul-Dec** of year X -> `X`

### Property portfolio
`property_portfolio_top_20` and `property_counts_by_country` come from the DB
`sgx_reit_property` table (matched by symbol + the calendar year the statement ends), NOT
the Excel sheet. If that period isn't in the property table yet, those keys are omitted.

## STEP 0 - install libraries (run once)

In [ ]:
!pip install -q supabase psycopg2-binary pandas openpyxl

## STEP 1 - tell it your files  *(the only cell you edit)*
- `EXCEL_FILES`: your workbook file name(s).
- `DRY_RUN = True` -> preview only. Set `False` to upload.

In [ ]:
EXCEL_FILES = [
    "v2 - SGX - FY 2024 - REIT.xlsx",
    "v2 - SGX - FY 2025 - REIT.xlsx",
]

DRY_RUN = True          # True = preview only. Set to False to upload.
ENV_PATH = ".env"       # holds SUPABASE_CONNECTION_STRING
TABLE = "sgx_manual_input"
PROPERTY_TABLE = "sgx_reit_property"
QUARTERLY_RATES_FILE = "quarterly_rates.json"  # MAS FX rates for non-SGD property values
PROPERTY_TOP_N = 20  # how many properties to keep in property_portfolio_top_20 (by SGD gross revenue)
TENANT_TABLE = "sgx_reit_top_tenant"    # GRI top-tenant breakdown (top_10_gri%_customers)
TRADEMIX_TABLE = "sgx_reit_trade_mix"   # GRI sector/trade-mix breakdown (gross_rental_income_by_sectors)
GRI_TOP_N = 10  # how many tenants to keep in top_10_gri%_customers (by GRI %)

## STEP 2 - read sheets + pull property from DB  *(no edits below)*

In [ ]:
import re, json, math
import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import Json, execute_values

_HEADER_LIKE = {"property destination", "Property Portfolio",
                "Breakdown of: top 20 property by Gross Revenue",
                "Breakdown of: property portfolio", "sum", "property category"}

# balance-sheet-area distribution rows (renamed for the DB); share counts handled separately
_DIST_FIELDS = {
    "distributable_income":          "Distributable income",
    "adjusted_distributable_income": "Adjusted distributable income",
    "distribution_paid":             "Distribution paid",
    "end_of_year_distribution":      "End-of-year distribution",
    "end_of_year_shareholder_units": "End-of-year shareholder units",
    "units_to_be_issued":            "Units to be issued",
}

def _load_env(path):
    raw = open(path, "rb").read()
    txt = None
    for enc in ("utf-8-sig", "utf-16", "utf-8"):
        try:
            t = raw.decode(enc)
            if "SUPABASE" in t:
                txt = t; break
        except Exception:
            continue
    if txt is None:
        raise RuntimeError("Could not read " + path)
    env = {}
    for l in txt.splitlines():
        l = l.strip().lstrip("\ufeff")
        if "=" in l and not l.startswith("#"):
            k, v = l.split("=", 1); env[k.strip()] = v.strip()
    return env

def none_value_extractor(m):
    return None if (m is None or (isinstance(m, float) and pd.isna(m))) else m

def _int_or_none(v):
    return int(v) if (v is not None and not (isinstance(v, float) and pd.isna(v))) else None

def _reduce_others_keys(df):
    others = df[df.index.str.contains("^[Oo]ther")]
    df = df[~df.index.str.contains("^[Oo]ther")]
    if others.shape[0] > 0:
        df.loc["Others", "value"] = int(others["value"].sum())
        df.loc["Others", "category"] = others["category"].iloc[0]
    return df

def _num(x):
    if not isinstance(x, str):
        return x
    s = x.strip()
    if s == "" or not any(ch.isdigit() for ch in s):
        return x
    if re.fullmatch(r"\(?\s*-?[\d,]+(?:\.\d+)?\s*\)?%?", s) is None:
        return x
    neg = s.startswith("(") and s.rstrip("%").endswith(")")
    core = s.replace("(", "").replace(")", "").replace(",", "").replace("%", "").strip()
    try:
        v = float(core)
    except ValueError:
        return x
    return -v if neg else v

def financial_year_from_date(date_str):
    d = pd.to_datetime(date_str)
    return int(d.year - 1 if d.month <= 6 else d.year)

def make_sankey_component(inp):
    light_blue, orange, dark_blue, red = "hsl(195, 53%, 79%)", "hsl(39, 100%, 50%)", "hsl(240, 100%, 50%)", "hsl(0, 100%, 50%)"
    links, nodes = [], []
    for rb in inp["revenue_breakdown"]:
        links.append({"source": rb["category"], "target": "Total Revenue", "value": rb["amount"]})
    if inp["gross_income"] >= 0:
        links.append({"source": "Total Revenue", "target": "Cost of Revenue", "value": inp["cost_of_revenue"]})
        links.append({"source": "Total Revenue", "target": "Gross Profit", "value": inp["gross_income"]})
        if inp["operating_income"] >= 0:
            links.append({"source": "Gross Profit", "target": "Operating Income", "value": inp["operating_income"]})
            links.append({"source": "Gross Profit", "target": "Operating Expense", "value": inp["operating_expense"]})
        else:
            links.append({"source": "Operating Income", "target": "Operating Expense", "value": -(inp["operating_income"])})
            links.append({"source": "Gross Profit", "target": "Operating Expense", "value": inp["gross_income"]})
    else:
        links.append({"source": "Total Revenue", "target": "Cost of Revenue", "value": inp["total_revenue"]})
        links.append({"source": "Gross Profit", "target": "Cost of Revenue", "value": -(inp["gross_income"])})
        links.append({"source": "Operating Income", "target": "Gross Profit", "value": -(inp["gross_income"])})
        links.append({"source": "Operating Income", "target": "Operating Expense", "value": inp["operating_expense"]})
    for oeb in inp["operating_expense_breakdown"]:
        links.append({"source": "Operating Expense", "target": oeb["category"], "value": oeb["amount"]})
    for rb in inp["revenue_breakdown"]:
        nodes.append({"id": rb["category"], "nodeColor": light_blue})
    nodes.append({"id": "Total Revenue", "nodeColor": light_blue})
    nodes.append({"id": "Cost of Revenue", "nodeColor": orange})
    nodes.append({"id": "Gross Profit", "nodeColor": dark_blue if inp["gross_income"] >= 0 else red})
    nodes.append({"id": "Operating Income", "nodeColor": dark_blue if inp["operating_income"] >= 0 else red})
    nodes.append({"id": "Operating Expense", "nodeColor": orange})
    for oeb in inp["operating_expense_breakdown"]:
        nodes.append({"id": oeb["category"], "nodeColor": orange})
    return {"nodes": nodes, "links": links}

def sanitize_for_json(o):
    if isinstance(o, dict):
        return {k: sanitize_for_json(v) for k, v in o.items()}
    if isinstance(o, list):
        return [sanitize_for_json(v) for v in o]
    if isinstance(o, float):
        return None if (math.isnan(o) or math.isinf(o)) else (int(o) if o.is_integer() else o)
    if isinstance(o, (np.generic,)):
        return sanitize_for_json(o.item())
    return o

def _load_rates(path):
    data = json.load(open(path, encoding="utf-8"))["quarters"]
    parsed = sorted((pd.to_datetime(k), k) for k in data)
    return data, parsed

def _nearest_quarter(parsed, target):
    t = pd.to_datetime(target)
    return min(parsed, key=lambda p: abs((p[0] - t).days))[1]

def _to_sgd(value, ccy, target_date, rates, parsed, warn=None):
    if value is None:
        return None
    if ccy in (None, "", "SGD"):
        return float(value)
    tbl = rates.get(_nearest_quarter(parsed, target_date), {})
    if ccy in tbl and "SGD" in tbl[ccy]:
        return float(value) * tbl[ccy]["SGD"]
    if warn is not None:
        warn.append("no FX rate for " + str(ccy) + " (left unconverted)")
    return float(value)

def property_from_db(cur, symbol, property_fy, rates, parsed, table="sgx_reit_property", fallback_date=None, warn=None, top_n=20):
    """Top-20 (by SGD gross revenue) + counts-by-country from the DB property table.
    Non-SGD gross_revenue / market_valuation are converted to SGD via quarterly MAS rates."""
    cur.execute(
        "select property_name, country, category, ownership, market_valuation, market_valuation_currency, "
        "gross_revenue, gross_revenue_currency, occupancy_rate, valuation_date "
        "from public." + table + " where symbol=%s and financial_year=%s",
        (symbol, property_fy))
    rows = cur.fetchall()
    if not rows:
        return None, None
    conv = []
    for name, country, cat, own, mv, mvc, gr, grc, occ, vdate in rows:
        d = vdate or fallback_date
        conv.append((name, country, cat, own,
                     _to_sgd(mv, mvc, d, rates, parsed, warn),
                     _to_sgd(gr, grc, d, rates, parsed, warn), occ))
    conv.sort(key=lambda r: (r[5] if r[5] is not None else float("-inf")), reverse=True)
    top = []
    for name, country, cat, own, mv, gr, occ in conv[:top_n]:
        e = {}
        if country is not None: e["country"] = country
        if cat is not None: e["category"] = cat
        if name is not None: e["name"] = name
        if own is not None and float(own) != 100: e["ownership_pct"] = round(float(own) / 100, 2)
        if mv is not None: e["valuation"] = int(round(mv))
        if gr is not None: e["gross_income"] = int(round(gr))
        if occ is not None: e["occupancy_rate"] = round(float(occ) / 100, 2)
        top.append(e)
    counts = {}
    for name, country, cat, own, mv, gr, occ in conv:
        c = country if country is not None else "Unknown"
        k = cat if cat is not None else "Unknown"
        slot = counts.setdefault(c, {}).setdefault(k, [0, 0, 0])
        slot[0] += 1
        slot[1] += int(round(gr)) if gr is not None else 0
        slot[2] += int(round(mv)) if mv is not None else 0
    return top, counts

def tenant_from_db(cur, symbol, fy, table="sgx_reit_top_tenant", limit=10):
    """Top tenants by GRI% from the DB -> list of {client_name, industry, revenue_pct}.
    Ordered by revenue_pct desc (nulls last), capped at `limit`. revenue_pct is 0-100 in the
    DB -> stored as a 0-1 fraction (2 dp) to match the Excel shape. Returns None if no rows."""
    cur.execute(
        "select rank, client_name, industry, revenue_pct from public." + table +
        " where symbol=%s and financial_year=%s", (symbol, fy))
    rows = cur.fetchall()
    if not rows:
        return None
    rows.sort(key=lambda r: (r[3] is None, -(float(r[3]) if r[3] is not None else 0.0), r[0]))
    out = []
    for rank, name, ind, pct in rows[:limit]:
        e = {}
        if name is not None: e["client_name"] = name
        if ind is not None: e["industry"] = ind
        e["revenue_pct"] = round(float(pct) / 100, 2) if pct is not None else None
        out.append(e)
    return out

def trademix_from_db(cur, symbol, fy, table="sgx_reit_trade_mix"):
    """Sector/trade-mix by GRI% from the DB -> {category: fraction}. The DB `category` is a
    normalised bucket that repeats (finer `category_raw` rows collapse into it), so pct is SUMMED
    per category. pct is 0-100 -> 0-1 fraction (2 dp). Sorted by value desc. Returns None if empty."""
    cur.execute(
        "select category, pct from public." + table +
        " where symbol=%s and financial_year=%s", (symbol, fy))
    rows = cur.fetchall()
    if not rows:
        return None
    agg = {}
    for cat, pct in rows:
        if cat is None or pct is None:
            continue
        agg[cat] = agg.get(cat, 0.0) + float(pct)
    if not agg:
        return None
    # Some REITs (e.g. T82U) publish TWO trade-mix tables (office + retail), each summing to ~100%,
    # with no field that separates them -> aggregating would give ~200%. Reject the ambiguous
    # multi-segment case and let the caller fall back to Excel.
    if sum(agg.values()) > 130:
        return None
    return {k: round(v / 100, 2) for k, v in sorted(agg.items(), key=lambda kv: -kv[1])}

def extract_reit(df):
    warn = []
    df = df.map(_num)

    date = pd.to_datetime(df.iloc[1, 4]).strftime("%Y-%m-%d")
    data = df[[0, 1]].rename(columns={0: "key", 1: "value"}).dropna(subset=["key"]).set_index("key")
    metadata = data.loc["symbol":"currency"].copy().replace({np.nan: None})
    currency = metadata.loc["currency"].value
    if currency != "SGD":
        warn.append("currency=" + str(currency) + " not SGD (values left unconverted)")

    income_stmt = data.loc["total revenue":"FFO"].copy()
    income_stmt["value"] = income_stmt["value"].fillna(0)

    revenue_bd = df[[2, 3, 4]].rename(columns={2: "category", 3: "key", 4: "value"}).dropna(subset=["key"])
    revenue_bd = revenue_bd.set_index("key").drop(index=["sum", "match", "Breakdown of: total revenue", "Date"], errors="ignore")
    revenue_bd = _reduce_others_keys(revenue_bd); revenue_bd.reset_index(inplace=True)

    expense_bd = df[[5, 6, 7]].rename(columns={5: "category", 6: "key", 7: "value"}).dropna(subset=["key"])
    expense_bd = expense_bd.set_index("key").drop(index=["sum", "match", "Breakdown of: operating expenses"], errors="ignore")
    if (expense_bd["value"].values > 0).any():
        warn.append("positive value in expense breakdown")
    expense_bd = _reduce_others_keys(expense_bd)
    for ei in list(expense_bd.index):
        if ei in revenue_bd["key"].values or ei == "Others":
            expense_bd.rename(index={ei: ei + " (expense)"}, inplace=True)
    expense_bd.reset_index(inplace=True); expense_bd["value"] = expense_bd["value"].astype("int")

    bal_sheet = df.iloc[:60, 13:15].copy(); bal_sheet.columns = ["key", "value"]; bal_sheet = bal_sheet.set_index("key")
    col13 = {}
    for k, v in zip(df[13], df[14]):
        if isinstance(k, str) and k.strip() and k.strip() not in col13:
            col13[k.strip()] = v
    def bs(name): return none_value_extractor(bal_sheet.loc[name].value) if name in bal_sheet.index else None
    def isv(name): return none_value_extractor(income_stmt.loc[name].value) if name in income_stmt.index else None

    basic = col13.get("Weighted average number of ordinary shares in issue (basic)")
    diluted = col13.get("Weighted average number of ordinary shares in issue (diluted)")
    if diluted is None:
        diluted = bs("Weighted average shares outstanding")

    ebit_cell = income_stmt.loc["ebit"].value
    ebitda_cell = income_stmt.loc["ebitda"].value
    dna = income_stmt.loc["depreciation and amortization"].value
    ebitda = ebit_cell + abs(dna) if pd.isna(ebitda_cell) else ebitda_cell

    input = {
        "symbol": metadata.loc["symbol"].value,
        "url": metadata.loc["url"].value,
        "total_revenue": isv("total revenue"),
        "cost_of_revenue": none_value_extractor(-(income_stmt.loc["cost of revenue"].value)),
        "gross_income": isv("gross income"),
        "operating_expense": none_value_extractor(-(income_stmt.loc["operating expenses"].value)),
        "operating_income": isv("net operating income"),
        "non_operating_income_or_loss": isv("net non operating income/(expenses)"),
        "pretax_income": isv("pretax income"),
        "income_taxes": none_value_extractor(-(income_stmt.loc["tax"].value)),
        "net_income": isv("net income"),
        "minorities": isv("minorities"),
        "perpetual_security_holders": isv("perpetual security holders"),
        "unitholders": isv("unitholders"),
        "interest_expense_non_operating": none_value_extractor(-(income_stmt.loc["non operating interest expense"].value)),
        "ebit": none_value_extractor(ebit_cell),
        "ebitda": none_value_extractor(ebitda),
        "net_property_sales": isv("gain/(loss) on property sales"),
        "funds_from_operation": isv("FFO"),
        "basic_shares_outstanding": none_value_extractor(basic),
        "diluted_shares_outstanding": none_value_extractor(diluted),
        "revenue_breakdown": [{"class": revenue_bd.iloc[i, 1], "category": revenue_bd.iloc[i, 0], "amount": revenue_bd.iloc[i, 2]} for i in range(revenue_bd.shape[0])],
        "operating_expense_breakdown": [{"class": expense_bd.iloc[i, 1], "category": expense_bd.iloc[i, 0], "amount": int(-(expense_bd.iloc[i, 2]))} for i in range(expense_bd.shape[0])],
    }

    balance_sheet = {
        "total_current_asset": bs("Current Asset"), "total_non_current_asset": bs("Non-Current Asset"),
        "total_asset": bs("TOTAL ASSET"), "total_current_liabilities": bs("Current Liabilities"),
        "total_non_current_liabilities": bs("Non-Current Liabilities"), "total_liabilities": bs("TOTAL LIABILITIES"),
        "total_equity": bs("TOTAL SHAREHOLDER'S EQUITY"), "working_capital": bs("Working Capital"),
    }

    capex = bs("CAPITAL EXPENDITURE")
    if capex is None and {"Net PP&E (current)", "Net PP&E (previous year)", "Depreciation expenses (current)"} <= set(bal_sheet.index):
        capex = bal_sheet.loc["Net PP&E (current)"].value - bal_sheet.loc["Net PP&E (previous year)"].value + bal_sheet.loc["Depreciation expenses (current)"].value
    ocf = bs("Cash Flows from Operating Activities")
    cash_flow = {
        "operating_cash_flow": ocf, "investing_cash_flow": bs("Cash Flows from Investing Activities"),
        "financing_cash_flow": bs("Cash Flows from Financing Activities"), "net_cash_flow": bs("NET INCREASE/DECREASED"),
        "capital_expenditure": none_value_extractor(capex),
        "free_cash_flow": none_value_extractor(ocf - capex if (ocf is not None and capex is not None) else None),
    }

    distribution_metrics = {k: _int_or_none(col13.get(label)) for k, label in _DIST_FIELDS.items()}

    for k in balance_sheet:
        balance_sheet[k] = int(balance_sheet[k]) if balance_sheet[k] is not None else None
    for k in ["total_revenue", "cost_of_revenue", "gross_income", "operating_expense", "operating_income",
              "non_operating_income_or_loss", "pretax_income", "income_taxes", "net_income", "minorities",
              "perpetual_security_holders", "unitholders", "interest_expense_non_operating", "ebit", "ebitda",
              "net_property_sales", "funds_from_operation", "basic_shares_outstanding", "diluted_shares_outstanding"]:
        input[k] = int(input[k]) if input[k] is not None else None
    for bd in ["revenue_breakdown", "operating_expense_breakdown"]:
        for it in input[bd]:
            it["amount"] = int(it["amount"])
    cash_flow = {k: (int(v) if v is not None else None) for k, v in cash_flow.items()}

    employee = None
    loc = np.argwhere(df.values == "TOTAL NUMBER OF EMPLOYEES")
    if len(loc):
        r, c = int(loc[0][0]), int(loc[0][1])
        total = df.iloc[r, c + 1]
        if pd.notna(total) and total != 0:
            def ev(rr):
                v = df.iloc[rr, c + 1]
                return int(v) if pd.notna(v) else None
            employee = {"permanent_employee": ev(r - 3), "contract_employee": ev(r - 2),
                        "others_employee": ev(r - 1), "total_employee": int(total)}

    # tenant / sector breakdown from Excel (property portfolio comes from the DB later)
    industry_breakdown = {}
    try:
        cust = df.iloc[5:, 9:12].copy()
        cust.columns = ["client_name", "industry", "revenue_pct"]; cust.reset_index(inplace=True, drop=True)
        g = cust.index[cust["client_name"] == "Breakdown of: by Gross Rental Income (GRI%)"]
        if len(g):
            g = g[0]
            sect = cust.iloc[g - 1:].copy(); sect = sect[(~sect.revenue_pct.isna()) & (sect.client_name != "sum")]
            sect["revenue_pct"] = (sect["revenue_pct"].astype("float") / 100).round(2)
            if sect.dropna(subset="industry").shape[0] == 0:
                sect = sect[["client_name", "revenue_pct"]]; sect.columns = ["sector", "revenue_pct"]
            industry_breakdown["gross_rental_income_by_sectors"] = dict(zip(sect["sector"], sect["revenue_pct"]))
            top10 = cust.iloc[:g - 1].copy(); top10 = top10[(~top10.revenue_pct.isna()) & (top10.client_name != "sum")]
            top10["revenue_pct"] = (top10["revenue_pct"].astype("float") / 100).round(2)
            if top10.dropna(subset="industry").shape[0] == 0:
                top10 = top10[["client_name", "revenue_pct"]]
            industry_breakdown["top_10_gri%_customers"] = top10.reset_index(drop=True).to_dict(orient="records")
        else:
            warn.append("no GRI customer breakdown")
    except Exception as e:
        warn.append("customer parse failed: " + str(e))

    industry_breakdown["distribution_metrics"] = distribution_metrics

    rev_sum = sum(r["amount"] for r in input["revenue_breakdown"])
    exp_sum = sum(e["amount"] for e in input["operating_expense_breakdown"])
    if input["total_revenue"] is not None and rev_sum != input["total_revenue"]:
        warn.append("revenue breakdown " + str(rev_sum) + " != total_revenue " + str(input["total_revenue"]))
    if input["operating_expense"] is not None and exp_sum != input["operating_expense"]:
        warn.append("expense breakdown " + str(exp_sum) + " != operating_expense " + str(input["operating_expense"]))

    income_copy = input.copy(); income_copy.pop("symbol"); income_copy.pop("url")
    record = {
        "symbol": input["symbol"],
        "financial_year": financial_year_from_date(date),
        "sankey_component": make_sankey_component(input),
        "source_url": input["url"],
        "income_stmt_metrics": income_copy,
        "balance_sheet_metrics": balance_sheet,
        "cash_flow_metrics": cash_flow,
        "employee_breakdown": none_value_extractor(employee),
        "industry_breakdown": industry_breakdown,
        "updated_on": pd.to_datetime("now").strftime("%Y-%m-%d %H:%M:%S"),
        "date": date,
    }
    return record, warn

print("Helper functions loaded.")

In [ ]:
env = _load_env(ENV_PATH)
_conn = psycopg2.connect(env["SUPABASE_CONNECTION_STRING"]); _cur = _conn.cursor()
_RATES, _PARSED = _load_rates(QUARTERLY_RATES_FILE)

records, seen, skipped = [], {}, []
print(f"{'FILE':32}{'SHEET':7}{'SYMBOL':9}{'FY':5}{'DATE':12}{'propFY':7}{'props':6} notes")
print("-" * 100)
for f in EXCEL_FILES:
    xl = pd.ExcelFile(f)
    for sh in xl.sheet_names:
        rec, warn = extract_reit(pd.read_excel(f, sheet_name=sh, header=None))
        # NOTE (correctness): sgx_reit_property uses date.year labeling, while sgx_manual_input
        # uses the Q1/Q2->x-1 rule. They are offset by 1 year for Jan-Jun (e.g. March) year-ends.
        # Match by year(statement date) so the SAME period lines up (N2IU manual FY2024=Mar-2025
        # -> property FY2025=Mar-2025). Do NOT join directly on financial_year unless the property
        # table is relabeled to the same rule. See NOTES.md.
        prop_fy = pd.to_datetime(rec["date"]).year
        top, counts = property_from_db(_cur, rec["symbol"], prop_fy, _RATES, _PARSED, PROPERTY_TABLE, fallback_date=rec["date"], warn=warn, top_n=PROPERTY_TOP_N)
        # rebuild industry_breakdown in the canonical key order
        # GRI tenant / sector breakdowns: prefer the DB (sgx_reit_top_tenant / sgx_reit_trade_mix),
        # matched by year(statement date) like property; fall back to the Excel values otherwise.
        db_tenant = tenant_from_db(_cur, rec["symbol"], prop_fy, TENANT_TABLE, limit=GRI_TOP_N)
        db_trademix = trademix_from_db(_cur, rec["symbol"], prop_fy, TRADEMIX_TABLE)
        ib = rec["industry_breakdown"]
        ordered = {}
        if db_tenant is not None:
            ordered["top_10_gri%_customers"] = db_tenant
        elif "top_10_gri%_customers" in ib:
            ordered["top_10_gri%_customers"] = ib["top_10_gri%_customers"]; warn.append("tenants<-Excel(FY" + str(prop_fy) + ")")
        if db_trademix is not None:
            ordered["gross_rental_income_by_sectors"] = db_trademix
        elif "gross_rental_income_by_sectors" in ib:
            ordered["gross_rental_income_by_sectors"] = ib["gross_rental_income_by_sectors"]; warn.append("sectors<-Excel(FY" + str(prop_fy) + ")")
        if top is not None: ordered["property_portfolio_top_20"] = top
        if counts is not None: ordered["property_counts_by_country"] = counts
        ordered["distribution_metrics"] = ib["distribution_metrics"]
        rec["industry_breakdown"] = ordered
        if top is None:
            warn.append("no property in DB for FY" + str(prop_fy))

        key = (rec["symbol"], rec["financial_year"])
        if key in seen:
            skipped.append((rec["symbol"], rec["financial_year"], f, sh)); warn.append("SKIPPED dup of " + seen[key])
        else:
            seen[key] = f + ":" + sh; records.append(rec)
        print(f"{f[:31]:32}{sh:7}{rec['symbol']:9}{rec['financial_year']:<5}{rec['date']:12}{prop_fy:<7}{(len(top) if top else 0):<6} {'; '.join(warn)}")

_cur.close(); _conn.close()
records = [sanitize_for_json(r) for r in records]
print(f"\nReady to upload: {len(records)} rows.")
if skipped:
    print("Skipped duplicates:", skipped)

## STEP 3 - preview

In [ ]:
prev = pd.DataFrame([{
    "symbol": r["symbol"], "financial_year": r["financial_year"], "date": r["date"],
    "total_revenue": r["income_stmt_metrics"].get("total_revenue"),
    "net_income": r["income_stmt_metrics"].get("net_income"),
    "basic_shares": r["income_stmt_metrics"].get("basic_shares_outstanding"),
    "diluted_shares": r["income_stmt_metrics"].get("diluted_shares_outstanding"),
    "distributable_income": r["industry_breakdown"]["distribution_metrics"].get("distributable_income"),
    "distribution_paid": r["industry_breakdown"]["distribution_metrics"].get("distribution_paid"),
    "top_props(DB)": len(r["industry_breakdown"].get("property_portfolio_top_20", [])),
    "ib_keys": ", ".join(r["industry_breakdown"].keys()),
} for r in records])
prev

## STEP 4 - upload
`DRY_RUN = True` prints only. Set `False` in STEP 1 + re-run STEP 1, then run this.
Distribution now lives inside `industry_breakdown`, so the old top-level
`distribution_metrics` column is dropped if present.

In [ ]:
_COLS = ["symbol", "financial_year", "sankey_component", "source_url", "income_stmt_metrics",
         "balance_sheet_metrics", "cash_flow_metrics", "employee_breakdown", "industry_breakdown",
         "updated_on", "date"]
_JSONB = {"sankey_component", "income_stmt_metrics", "balance_sheet_metrics", "cash_flow_metrics",
          "employee_breakdown", "industry_breakdown"}

if DRY_RUN:
    print("DRY RUN - nothing uploaded. Would upsert", len(records), "rows into", TABLE + ".")
    print("Set DRY_RUN = False in STEP 1 (and re-run STEP 1) to upload.")
else:
    env = _load_env(ENV_PATH)
    conn = psycopg2.connect(env["SUPABASE_CONNECTION_STRING"]); cur = conn.cursor()
    cur.execute("""
    create table if not exists public.""" + TABLE + """ (
      symbol text not null, financial_year smallint not null,
      sankey_component jsonb, source_url text,
      income_stmt_metrics jsonb, balance_sheet_metrics jsonb, cash_flow_metrics jsonb,
      employee_breakdown jsonb, industry_breakdown jsonb,
      updated_on timestamptz not null default now(), date date,
      constraint """ + TABLE + """_pkey primary key (symbol, financial_year));""")
    cur.execute("alter table public." + TABLE + " drop column if exists distribution_metrics;")

    def _row(r):
        return tuple(Json(r.get(c)) if (c in _JSONB and r.get(c) is not None) else r.get(c) for c in _COLS)
    upd = ", ".join(c + "=excluded." + c for c in _COLS if c not in ("symbol", "financial_year"))
    sql = ("insert into public." + TABLE + " (" + ", ".join(_COLS) + ") values %s "
           "on conflict (symbol, financial_year) do update set " + upd)
    execute_values(cur, sql, [_row(r) for r in records])
    conn.commit()
    cur.execute("select count(*) from public." + TABLE + ";")
    print("Upload done. Table now has", cur.fetchone()[0], "rows.")
    cur.execute("select symbol, financial_year, date from public." + TABLE + " order by symbol, financial_year;")
    for row in cur.fetchall():
        print("   ", row[0], row[1], row[2])
    cur.close(); conn.close()